# Hands-on: One Base Model, Two LoRA Adapters

A main benefit of LoRA is that **one frozen base model can host many small task-specific adapters**, and you can switch between them at runtime.

| Role | Hugging Face ID | What it does |
|---|---|---|
| Base model | `Qwen/Qwen2.5-0.5B-Instruct` | General chat model (~0.5B params, ~1 GB) |
| Adapter 1 | `ali4566544/qwen2.5-0.5b-instruct-python-lora` | Python coding assistant |
| Adapter 2 | `9mark9/fin-lora-news-qwen2.5-0.5b` | Financial news sentiment |

```
                ┌── adapter "python"  → code answers
Qwen2.5-0.5B ───┤
 (frozen W)     └── adapter "finance" → sentiment labels
```

**Steps:** run the base model → load both adapters → switch between the adapters → compare the outputs.

**SageMaker setup:** use a PyTorch / Python 3 kernel. `ml.g5.xlarge` (GPU) is recommended, and `ml.t3.xlarge` (CPU) works but is slower. The notebook needs internet access to reach the Hugging Face Hub.

> Both adapters are community uploads. If one fails to load or gives weak results, replace its ID in the **Configuration** cell with any other adapter whose `base_model` is `Qwen/Qwen2.5-0.5B-Instruct`.

## 1. Install dependencies
Restart the kernel after this cell finishes.

In [ ]:
!pip install -q -U transformers peft accelerate

## 2. Imports and device setup

This cell loads the libraries, prints their versions, and decides **where** (GPU or CPU) and **in what precision** the model will run.

### Libraries

| Import | Library | Used for |
|---|---|---|
| `AutoModelForCausalLM` | 🤗 Transformers | Loads a text-generation model (one that predicts the next token) |
| `AutoTokenizer` | 🤗 Transformers | Converts text to token IDs and back |
| `PeftModel` | 🤗 PEFT | Attaches LoRA adapters to the base model |

The `Auto...` classes read the model's config on the Hugging Face Hub and pick the right architecture (here, Qwen2) automatically.

### Version check
Prints the installed versions of `torch`, `transformers` and `peft`. This is useful for troubleshooting, because many errors come from version mismatches.

### Device and precision

| Setting | GPU available | No GPU |
|---|---|---|
| `DEVICE` | `"cuda"` | `"cpu"` |
| `DTYPE` | `float16`: half the memory, fast on GPU | `float32`: the safe, fast default on most CPUs |

`DEVICE` and `DTYPE` are reused in later cells when loading the model and preparing the inputs.

**What to expect:** on a GPU instance (e.g. `ml.g5.xlarge`) you should see `Device: cuda`. If you see `cpu` on a GPU instance, check that you chose a GPU-enabled kernel.

In [ ]:
import torch, transformers, peft, gc
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

print("torch", torch.__version__, "| transformers", transformers.__version__, "| peft", peft.__version__)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32
print("Device:", DEVICE)

## 3. Configuration
Each adapter gets a **short name** that you use to switch to it later. There is one prompt per domain, and **every prompt goes to every model**, so you can see how each adapter specialises.

In [ ]:
BASE_ID = "Qwen/Qwen2.5-0.5B-Instruct"   # ~0.5B params, Apache-2.0

# Two independent LoRA adapters trained on the SAME base model
ADAPTERS = {
    "python": "ali4566544/qwen2.5-0.5b-instruct-python-lora",  # Python coding assistant
    "finance": "9mark9/fin-lora-news-qwen2.5-0.5b",            # Financial news sentiment
}

PROMPTS = {
    "python": "Write a Python function that checks if a string is a palindrome.",
    "finance": "Classify the sentiment of this financial headline as positive, negative or neutral: "
               "'Company X shares plunge 15% after profit warning.'",
}

## 4. Helper functions

This cell loads the **tokenizer** and defines two helpers that every later cell uses.

### Tokenizer
`AutoTokenizer` converts text into **token IDs** (numbers the model understands) and converts the model's output back into text.

### `generate(model, prompt, max_new_tokens=150)`
Sends one prompt to a model and returns only its answer. It works the same way for the **base model** and for **any LoRA adapter**, so the results can be compared fairly.

### `show(title, text)`
Prints each result under a clear heading so outputs from different models are easy to tell apart.

In [ ]:
tok = AutoTokenizer.from_pretrained(BASE_ID)

def generate(model, prompt, max_new_tokens=150):
    """Chat-template + greedy decoding -> deterministic, comparable outputs."""
    msgs = [{"role": "user", "content": prompt}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def show(title, text):
    print(f"\n{'=' * 25} {title} {'=' * 25}\n{text}")

## 5. Baseline: run the base model (no adapters)

Before adding any LoRA adapter, we record how the **original model** answers each prompt. This is our **reference point**: every adapter's output is compared against it later.

### Load the model
`AutoModelForCausalLM.from_pretrained(BASE_ID, ...)` downloads and loads the Qwen base model.

### Run the prompts
- `results` is a dictionary that stores every answer, organised as `results[prompt][model]`.
- The loop sends each prompt (`python`, `finance`) to the base model and saves the answer under the key `"base"`.
- Adapter answers are added to the same dictionary later, so step 8 can show them side by side in one table.

**What to expect:** the base model gives reasonable, general-purpose answers. It hasn't been specialised for coding or financial sentiment yet.

In [ ]:
base = AutoModelForCausalLM.from_pretrained(BASE_ID, torch_dtype=DTYPE).to(DEVICE).eval()
print(f"Base parameters: {sum(p.numel() for p in base.parameters())/1e6:.0f}M")

results = {name: {} for name in PROMPTS}
for task, prompt in PROMPTS.items():
    results[task]["base"] = generate(base, prompt)
    show(f"BASE | {task} prompt", results[task]["base"])

## 6. Load both LoRA adapters onto the same base model

Now we attach **two different fine-tuned "skills"** to the base model we already loaded. The base weights stay **frozen and in memory only once**. Each adapter adds only a few MB of extra weights.

### How the adapters are loaded

| Code | What it does |
|---|---|
| `PeftModel.from_pretrained(base, ..., adapter_name="python")` | Wraps the base model in a `PeftModel` and loads the **first** adapter under the name `"python"` |
| `model.load_adapter(..., adapter_name="finance")` | Adds a **second** adapter to the same wrapper under the name `"finance"` (nothing is copied or reloaded) |
| `model.eval()` | Switches to inference mode |

The `adapter_name` is just a label you choose. You use it later to switch between adapters with `set_adapter("python")` or `set_adapter("finance")`.

### Inspect each adapter's settings
The loop prints the LoRA settings stored in each adapter's config:

- **`r` (rank):** size of the low-rank matrices A and B. A smaller `r` means fewer trainable parameters (typical values: 8–64).
- **`alpha`:** scaling factor. The adapter's update is multiplied by `alpha / r`, which controls how strongly it changes the base model.
- **`targets`:** which layers of the model received adapter matrices (e.g. `q_proj` and `v_proj` in attention).

**What to expect:** both adapters appear in the "Loaded adapters" list. Their `r`, `alpha` and `targets` may differ because different people trained them with different settings.

In [ ]:
# First adapter creates the PeftModel wrapper...
model = PeftModel.from_pretrained(base, ADAPTERS["python"], adapter_name="python")
# ...additional adapters are loaded into the same wrapper by name
model.load_adapter(ADAPTERS["finance"], adapter_name="finance")
model.eval()

print("Loaded adapters:", list(model.peft_config.keys()))
for name, cfg in model.peft_config.items():
    print(f"  {name:8s} r={cfg.r}  alpha={cfg.lora_alpha}  targets={sorted(cfg.target_modules)}")

## 7. Switch between adapters and run every prompt

This is the core of the demo: **one model, two skills, switched at runtime**.

### How switching works
`model.set_adapter("python")` makes the `python` adapter's LoRA matrices active. All other adapters stay loaded in memory but are **ignored** until you switch to them.

- **Instant:** nothing is downloaded, reloaded or copied. Only the active adapter changes.
- **Safe:** the frozen base weights are never modified, so you can switch back and forth as often as you like.
- `model.active_adapter` shows which adapter is currently in use.

### What the loop does
| Loop | What happens |
|---|---|
| Outer: `for adapter in ADAPTERS` | Activates each adapter in turn (`python`, then `finance`) |
| Inner: `for task, prompt in PROMPTS.items()` | Sends **both** prompts to the active adapter |
| `results[task][adapter] = ...` | Saves each answer next to the base answer from step 5 |

That makes **2 adapters × 2 prompts = 4 answers**. Together with the 2 base answers, that's 6 results for the comparison table in step 8.

### Why send *every* prompt to *every* adapter?
- **On its own domain** (python adapter → coding prompt), the adapter should do **better** or answer in a more specialised style than the base model.
- **On the other domain** (python adapter → finance prompt), the answer shows whether the adapter still behaves generally or pulls everything toward its own task.

**What to expect:** the clearest changes usually show up on each adapter's own task. For example, the finance adapter may answer the headline prompt with a short sentiment label instead of a long explanation.

In [ ]:
for adapter in ADAPTERS:
    model.set_adapter(adapter)  # instant switch: no reload, base weights untouched
    print(f"\n######## ACTIVE ADAPTER: {model.active_adapter} ########")
    for task, prompt in PROMPTS.items():
        results[task][adapter] = generate(model, prompt)
        show(f"{adapter} adapter | {task} prompt", results[task][adapter])

## 8. Side-by-side comparison
Rows are prompts and columns are models. Compare how each adapter answers its **own** domain prompt with how it answers the **other** domain's prompt.

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", 300)
df = pd.DataFrame(results).T[["base", "python", "finance"]]
df.index.name = "prompt"
df

## 9. Clean up

In [ ]:
del model, base
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()